In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DecimalType, StructType, StructField, StringType, DoubleType, TimestampType, DateType

In [ ]:
# CONFIG

BRONZE_TABLE = "BRONZE.CAGED"
SILVER_SCHEMA = "SILVER"
SILVER_TABLE = "CAGED"
CONTROL_TABLE = "control_caged_quality"

SILVER_IDENTIFIER = f"{SILVER_SCHEMA}.{SILVER_TABLE}"
CONTROL_IDENTIFIER = f"{SILVER_SCHEMA}.{CONTROL_TABLE}"
COMPETENCIA_COL = "competênciamov"

# Quality gate.
FAIL_ON_INTEGER_CONVERSION_ERRORS = True
FAIL_ON_DECIMAL_CONVERSION_ERRORS = True
FAIL_ON_DATE_CONVERSION_ERRORS = True

print("Origem   :", BRONZE_TABLE)
print("Destino  :", SILVER_IDENTIFIER)
print("Controle :", CONTROL_IDENTIFIER)

In [ ]:
# INFRAESTRUTURA

# schema.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

# Tabela de controle + qualidade.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CONTROL_IDENTIFIER} (
    execution_id STRING,
    source_table STRING,
    target_table STRING,
    competencia_mov DATE,
    execution_started_at TIMESTAMP,
    processed_at TIMESTAMP,
    status STRING,
    metric_group STRING,
    metric_name STRING,
    column_name STRING,
    metric_value DOUBLE,
    metric_unit STRING,
    metric_status STRING,
    details STRING,
    error_message STRING
)
USING DELTA
""")

existing_control_columns = {f.name for f in spark.table(CONTROL_IDENTIFIER).schema.fields}
if "competencia_mov" not in existing_control_columns:
    spark.sql(f"ALTER TABLE {CONTROL_IDENTIFIER} ADD COLUMNS (competencia_mov DATE)")
    print(f"[MIGRAÇÃO] Coluna competencia_mov adicionada a {CONTROL_IDENTIFIER}.")

silver_exists = spark.catalog.tableExists(SILVER_IDENTIFIER)
print(f"Schema {SILVER_SCHEMA}: OK")
print(f"Tabela {CONTROL_IDENTIFIER}: OK")
print(f"Tabela {SILVER_IDENTIFIER}: {'EXISTE' if silver_exists else 'SERÁ CRIADA NA PRIMEIRA CARGA'}")

In [ ]:
# FUNÇÕES DE TRATAMENTO

def tratar_integer(coluna):
    valor = F.trim(F.col(coluna).cast("string"))
    return (
        F.when(valor == "", F.lit(None).cast(IntegerType()))
         .otherwise(valor.cast(IntegerType()))
    )


def tratar_decimal(coluna, precisao, escala):
    valor = F.trim(F.col(coluna).cast("string"))
    valor_normalizado = (
        F.when(valor == "", F.lit(None).cast("string"))
         .when(
             valor.contains(","),
             F.regexp_replace(
                 F.regexp_replace(valor, r"\.", ""),
                 ",",
                 "."
             )
         )
         .otherwise(valor)
    )
    return valor_normalizado.cast(DecimalType(precisao, escala))


def tratar_data_yyyymm(coluna):
    valor = F.trim(F.col(coluna).cast("string"))
    return F.to_date(valor, "yyyyMM")

In [ ]:
# PRÉ-VALIDAÇÃO: COMPETÊNCIAS DISPONÍVEIS NA BRONZE

execution_id = str(uuid4())
execution_started_at = datetime.now(timezone.utc)

df_bronze = spark.table(BRONZE_TABLE)

# A competência é mensal e será representada como o primeiro dia do mês.
df_bronze_comp = (
    df_bronze
    .select(
        tratar_data_yyyymm(COMPETENCIA_COL).alias("competencia_mov")
    )
    .dropDuplicates()
)

bronze_competencias = sorted({
    r["competencia_mov"]
    for r in df_bronze_comp.filter(F.col("competencia_mov").isNotNull()).collect()
})

invalid_competencia_rows = (
    df_bronze.filter(
        F.col(COMPETENCIA_COL).isNull() |
        (F.trim(F.col(COMPETENCIA_COL).cast("string")) == "") |
        tratar_data_yyyymm(COMPETENCIA_COL).isNull()
    ).count()
)

if invalid_competencia_rows:
    print(f"[WARN] Linhas com competência inválida/nula na Bronze: {invalid_competencia_rows:,}")

print(f"Competências válidas na Bronze: {bronze_competencias}")

In [ ]:
# PRÉ-VALIDAÇÃO: O QUE JÁ FOI PROCESSADO?


silver_competencias = set()
if silver_exists:
    silver_competencias = {
        r["competencia_mov"]
        for r in (
            spark.table(SILVER_IDENTIFIER)
                 .select(F.col(COMPETENCIA_COL).alias("competencia_mov"))
                 .filter(F.col("competencia_mov").isNotNull())
                 .dropDuplicates()
                 .collect()
        )
    }

control_success_competencias = {
    r["competencia_mov"]
    for r in (
        spark.table(CONTROL_IDENTIFIER)
             .filter(
                 (F.col("status") == "SUCCESS") &
                 (F.col("metric_group") == "execution") &
                 (F.col("metric_name") == "load_success") &
                 F.col("competencia_mov").isNotNull()
             )
             .select("competencia_mov")
             .dropDuplicates()
             .collect()
    )
}

processed_set = silver_competencias.union(control_success_competencias)
pending_competencias = sorted(set(bronze_competencias) - processed_set)
skipped_competencias = sorted(set(bronze_competencias).intersection(processed_set))

print("Já processadas:", skipped_competencias)
print("Pendentes     :", pending_competencias)

In [ ]:
# TRATAMENTO SILVER

def build_silver(df):
    return df.select(
        tratar_data_yyyymm("competênciamov").alias("competênciamov"),
        tratar_integer("região").alias("região"),
        tratar_integer("uf").alias("uf"),
        tratar_integer("município").alias("município"),
        F.substring(F.trim(F.col("seção")), 1, 1).alias("seção"),
        tratar_integer("subclasse").alias("subclasse"),
        tratar_integer("saldomovimentação").alias("saldomovimentação"),
        tratar_integer("cbo2002ocupação").alias("cbo2002ocupação"),
        tratar_integer("categoria").alias("categoria"),
        tratar_integer("graudeinstrução").alias("graudeinstrução"),
        tratar_integer("idade").alias("idade"),
        tratar_decimal("horascontratuais", 10, 2).alias("horascontratuais"),
        tratar_integer("raçacor").alias("raçacor"),
        tratar_integer("sexo").alias("sexo"),
        tratar_integer("tipoempregador").alias("tipoempregador"),
        tratar_integer("tipoestabelecimento").alias("tipoestabelecimento"),
        tratar_integer("tipomovimentação").alias("tipomovimentação"),
        tratar_integer("tipodedeficiência").alias("tipodedeficiência"),
        tratar_integer("indtrabintermitente").alias("indtrabintermitente"),
        tratar_integer("indtrabparcial").alias("indtrabparcial"),
        tratar_decimal("salário", 18, 2).alias("salário"),
        tratar_integer("tamestabjan").alias("tamestabjan"),
        tratar_integer("indicadoraprendiz").alias("indicadoraprendiz"),
        tratar_integer("origemdainformação").alias("origemdainformação"),
        tratar_integer("competênciadec").alias("competênciadec"),
        tratar_integer("indicadordeforadoprazo").alias("indicadordeforadoprazo"),
        tratar_integer("unidadesaláriocódigo").alias("unidadesaláriocódigo"),
        tratar_decimal("valorsaláriofixo", 18, 2).alias("valorsaláriofixo"),
        F.col("_source_archive"),
        F.col("_source_file"),
        F.col("_ingestion_timestamp")
    )

In [ ]:
# PROCESSAMENTO POR COMPETÊNCIA

integer_columns = [
    "região", "uf", "município", "subclasse", "saldomovimentação",
    "cbo2002ocupação", "categoria", "graudeinstrução", "idade", "raçacor",
    "sexo", "tipoempregador", "tipoestabelecimento", "tipomovimentação",
    "tipodedeficiência", "indtrabintermitente", "indtrabparcial", "tamestabjan",
    "indicadoraprendiz", "origemdainformação", "competênciadec",
    "indicadordeforadoprazo", "unidadesaláriocódigo"
]

decimal_columns = {
    "horascontratuais": (10, 2),
    "salário": (18, 2),
    "valorsaláriofixo": (18, 2),
}

date_columns = ["competênciamov"]
string_trim_columns = ["seção"]

metric_schema = StructType([
    StructField("execution_id", StringType(), False),
    StructField("source_table", StringType(), False),
    StructField("target_table", StringType(), False),
    StructField("competencia_mov", DateType(), True),
    StructField("execution_started_at", TimestampType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("metric_group", StringType(), True),
    StructField("metric_name", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("metric_value", DoubleType(), True),
    StructField("metric_unit", StringType(), True),
    StructField("metric_status", StringType(), True),
    StructField("details", StringType(), True),
    StructField("error_message", StringType(), True),
])


def write_metrics(rows):
    if not rows:
        return
    (
        spark.createDataFrame(rows, schema=metric_schema)
        .write.format("delta")
        .mode("append")
        .saveAsTable(CONTROL_IDENTIFIER)
    )


def metric_row(competencia, group, name, column, value, unit="count", metric_status="INFO", details=None, status="SUCCESS", error_message=None):
    return (
        execution_id, BRONZE_TABLE, SILVER_IDENTIFIER, competencia,
        execution_started_at, datetime.now(timezone.utc), status,
        group, name, column,
        float(value) if value is not None else None,
        unit, metric_status, details, error_message
    )


for competencia in pending_competencias:
    print(f"\n========== PROCESSANDO {competencia} ==========")

    # Revalidação por competência imediatamente antes do processamento.
    already_in_silver = False
    if spark.catalog.tableExists(SILVER_IDENTIFIER):
        already_in_silver = (
            spark.table(SILVER_IDENTIFIER)
                 .filter(F.col(COMPETENCIA_COL) == F.lit(competencia))
                 .limit(1)
                 .count() > 0
        )

    already_success = (
        spark.table(CONTROL_IDENTIFIER)
             .filter(
                 (F.col("status") == "SUCCESS") &
                 (F.col("metric_group") == "execution") &
                 (F.col("metric_name") == "load_success") &
                 (F.col("competencia_mov") == F.lit(competencia))
             )
             .limit(1)
             .count() > 0
    )

    if already_in_silver or already_success:
        write_metrics([
            metric_row(
                competencia, "execution", "competencia_already_processed", COMPETENCIA_COL,
                1, "flag", "SKIP",
                "Revalidação final: competência já está na Silver ou possui load_success no controle.",
                status="SKIPPED"
            )
        ])
        print(f"[SKIP] {competencia} já processada.")
        continue

    try:
        # Seleciona somente o mês atual. Nenhum outro mês entra na transformação.
        df_comp = (
            df_bronze
            .withColumn("__competencia_mov", tratar_data_yyyymm(COMPETENCIA_COL))
            .filter(F.col("__competencia_mov") == F.lit(competencia))
            .drop("__competencia_mov")
        )

        df_silver = build_silver(df_comp)

        # Uma única agregação por competência calcula as principais métricas.
        metric_aggs = [F.count(F.lit(1)).cast("double").alias("__rows")]

        for col in df_silver.columns:
            if col.startswith("_"):
                continue
            metric_aggs.append(
                F.sum(F.when(F.col(col).isNull(), 1).otherwise(0)).cast("double").alias(f"__nulls__{col}")
            )

        def raw_string(col):
            return F.trim(F.col(col).cast("string"))

        def raw_nonblank(col):
            raw = F.col(col).cast("string")
            return F.col(col).isNotNull() & (F.trim(raw) != "")

        for col in integer_columns:
            raw = raw_string(col)
            converted = tratar_integer(col)
            metric_aggs += [
                F.sum(F.when(raw == "", 1).otherwise(0)).cast("double").alias(f"__integer_blank__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNotNull(), 1).otherwise(0)).cast("double").alias(f"__integer_converted__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNull(), 1).otherwise(0)).cast("double").alias(f"__integer_failed__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNotNull() & (raw != converted.cast("string")), 1).otherwise(0)).cast("double").alias(f"__integer_changed__{col}"),
            ]

        for col, (precision, scale) in decimal_columns.items():
            raw = raw_string(col)
            converted = tratar_decimal(col, precision, scale)
            metric_aggs += [
                F.sum(F.when(raw == "", 1).otherwise(0)).cast("double").alias(f"__decimal_blank__{col}"),
                F.sum(F.when(raw_nonblank(col) & raw.contains(","), 1).otherwise(0)).cast("double").alias(f"__decimal_comma__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNotNull(), 1).otherwise(0)).cast("double").alias(f"__decimal_converted__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNull(), 1).otherwise(0)).cast("double").alias(f"__decimal_failed__{col}"),
            ]

        for col in date_columns:
            raw = raw_string(col)
            converted = tratar_data_yyyymm(col)
            metric_aggs += [
                F.sum(F.when(raw == "", 1).otherwise(0)).cast("double").alias(f"__date_blank__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNotNull(), 1).otherwise(0)).cast("double").alias(f"__date_valid__{col}"),
                F.sum(F.when(raw_nonblank(col) & converted.isNull(), 1).otherwise(0)).cast("double").alias(f"__date_failed__{col}"),
            ]

        for col in string_trim_columns:
            original = F.col(col).cast("string")
            cleaned = F.substring(F.trim(original), 1, 1)
            metric_aggs += [
                F.sum(F.when(original.isNotNull() & (original != cleaned), 1).otherwise(0)).cast("double").alias(f"__string_changed__{col}"),
                F.sum(F.when(F.trim(original) == "", 1).otherwise(0)).cast("double").alias(f"__string_blank__{col}"),
            ]

        affected = []
        for col in integer_columns:
            raw = raw_string(col); converted = tratar_integer(col)
            affected += [raw == "", raw_nonblank(col) & converted.isNull(), raw_nonblank(col) & converted.isNotNull() & (raw != converted.cast("string"))]
        for col, (precision, scale) in decimal_columns.items():
            raw = raw_string(col); converted = tratar_decimal(col, precision, scale)
            affected += [raw == "", raw_nonblank(col) & raw.contains(","), raw_nonblank(col) & converted.isNull()]
        for col in date_columns:
            raw = raw_string(col); converted = tratar_data_yyyymm(col)
            affected += [raw == "", raw_nonblank(col) & converted.isNull()]
        for col in string_trim_columns:
            original = F.col(col).cast("string"); cleaned = F.substring(F.trim(original), 1, 1)
            affected.append(original.isNotNull() & (original != cleaned))

        affected_expr = affected[0]
        for condition in affected[1:]:
            affected_expr = affected_expr | condition
        metric_aggs.append(F.sum(F.when(affected_expr, 1).otherwise(0)).cast("double").alias("__rows_affected_by_cleaning"))

        quality = df_comp.agg(*metric_aggs).collect()[0].asDict()
        rows_processed = int(quality["__rows"] or 0)

        integer_failures = sum(int(quality.get(f"__integer_failed__{c}") or 0) for c in integer_columns)
        decimal_failures = sum(int(quality.get(f"__decimal_failed__{c}") or 0) for c in decimal_columns)
        date_failures = sum(int(quality.get(f"__date_failed__{c}") or 0) for c in date_columns)

        quality_gate_failed = (
            (FAIL_ON_INTEGER_CONVERSION_ERRORS and integer_failures > 0) or
            (FAIL_ON_DECIMAL_CONVERSION_ERRORS and decimal_failures > 0) or
            (FAIL_ON_DATE_CONVERSION_ERRORS and date_failures > 0)
        )

        if quality_gate_failed:
            gate_details = (
                f"integer_failures={integer_failures}; "
                f"decimal_failures={decimal_failures}; "
                f"date_failures={date_failures}; "
            )
            write_metrics([
                metric_row(
                    competencia, "quality_gate", "load_blocked", None, 1, "flag", "ERROR",
                    "Carga bloqueada antes da Silver por falha nas regras críticas de qualidade: " + gate_details,
                    status="ERROR", error_message=gate_details
                )
            ])
            raise RuntimeError("Quality gate reprovado. Nenhuma linha da competência foi gravada na Silver.")

        # Criação explícita da Silver quando ainda não existe.
        if not spark.catalog.tableExists(SILVER_IDENTIFIER):
            (
                df_silver.limit(0)
                .write.format("delta")
                .mode("overwrite")
                .saveAsTable(SILVER_IDENTIFIER)
            )
            silver_exists = True
            print(f"[OK] Tabela {SILVER_IDENTIFIER} criada.")

        # Revalidação imediatamente antes da escrita.
        race_check = (
            spark.table(SILVER_IDENTIFIER)
                 .filter(F.col(COMPETENCIA_COL) == F.lit(competencia))
                 .limit(1)
                 .count()
        )
        if race_check:
            raise RuntimeError(f"A competência {competencia} apareceu na Silver antes da escrita; carga abortada para evitar duplicidade.")

        # APPEND, nunca OVERWRITE.
        df_silver.write.format("delta").mode("append").saveAsTable(SILVER_IDENTIFIER)

        metric_rows = [
            metric_row(competencia, "execution", "rows_processed", None, rows_processed, "rows"),
            metric_row(competencia, "execution", "load_success", None, 1, "flag", "OK", "Carga mensal concluída com sucesso."),
            metric_row(competencia, "execution", "invalid_competencia_rows", COMPETENCIA_COL, 0, "rows", "OK"),
            metric_row(competencia, "transformation", "rows_affected_by_cleaning", None, quality.get("__rows_affected_by_cleaning") or 0, "rows"),
        ]

        for col in df_silver.columns:
            if col.startswith("_"):
                continue
            nulls = quality.get(f"__nulls__{col}") or 0
            pct = 100.0 * nulls / rows_processed if rows_processed else 0.0
            metric_rows += [
                metric_row(competencia, "completeness", "null_count", col, nulls, "rows", "WARN" if nulls else "OK"),
                metric_row(competencia, "completeness", "null_pct", col, pct, "percent", "WARN" if nulls else "OK"),
            ]

        for col in integer_columns:
            failed = quality.get(f"__integer_failed__{col}") or 0
            metric_rows += [
                metric_row(competencia, "transformation", "integer_blank_to_null", col, quality.get(f"__integer_blank__{col}") or 0),
                metric_row(competencia, "transformation", "integer_representation_changed", col, quality.get(f"__integer_changed__{col}") or 0),
                metric_row(competencia, "transformation", "integer_conversion_failed", col, failed, "rows", "ERROR" if failed else "OK"),
            ]

        for col in decimal_columns:
            failed = quality.get(f"__decimal_failed__{col}") or 0
            metric_rows += [
                metric_row(competencia, "transformation", "decimal_comma_normalized", col, quality.get(f"__decimal_comma__{col}") or 0),
                metric_row(competencia, "transformation", "decimal_conversion_failed", col, failed, "rows", "ERROR" if failed else "OK"),
            ]

        for col in date_columns:
            failed = quality.get(f"__date_failed__{col}") or 0
            metric_rows += [
                metric_row(competencia, "transformation", "date_valid", col, quality.get(f"__date_valid__{col}") or 0),
                metric_row(competencia, "transformation", "date_conversion_failed", col, failed, "rows", "ERROR" if failed else "OK"),
            ]

        for col in ["salário", "valorsaláriofixo"]:
            below = quality.get(f"__salary_below_min__{col}") or 0
            above = quality.get(f"__salary_above_max__{col}") or 0
            metric_rows += [
                metric_row(competencia, "validity", "below_min_count", col, below, "rows", "WARN" if below else "OK"),
                metric_row(competencia, "validity", "above_max_count", col, above, "rows", "WARN" if above else "OK"),
                metric_row(competencia, "distribution", "min_value", col, quality.get(f"__min__{col}"), "value"),
                metric_row(competencia, "distribution", "max_value", col, quality.get(f"__max__{col}"), "value"),
            ]

        write_metrics(metric_rows)
        print(f"[OK] {competencia}: {rows_processed:,} linhas carregadas e métricas registradas.")

    except Exception as e:
        write_metrics([
            metric_row(
                competencia, "execution", "load_error", None, 0, "flag", "ERROR",
                "Carga abortada. A competência não deve ser considerada processada.",
                status="ERROR", error_message=str(e)
            )
        ])
        print(f"[ERRO] {competencia}: {e}")
        raise

# Registra as competências que já estavam processadas na primeira pré-validação.
for competencia in skipped_competencias:
    write_metrics([
        metric_row(
            competencia, "execution", "competencia_already_processed", COMPETENCIA_COL,
            1, "flag", "SKIP",
            "Competência já existente na Silver ou marcada como load_success no controle.",
            status="SKIPPED"
        )
    ])

if not pending_competencias:
    print("Nenhuma competência nova. Nenhuma linha foi escrita na Silver.")

In [ ]:
# DIAGNÓSTICO DOS ERROS DA EXECUÇÃO

display(
    spark.table(CONTROL_IDENTIFIER)
        .filter(
            (F.col("execution_id") == execution_id) &
            (
                (F.col("metric_status") == "ERROR") |
                (F.col("metric_group") == "quality_gate")
            )
        )
        .select(
            "competencia_mov",
            "metric_group",
            "metric_name",
            "column_name",
            "metric_value",
            "metric_status",
            "details",
            "error_message"
        )
        .withColumn(
            "erro_identificado",
            F.when(
                F.col("error_message").contains("integer_failures=") &
                ~F.col("error_message").contains("integer_failures=0"),
                "CONVERSÃO INTEGER"
            )
            .when(
                F.col("error_message").contains("decimal_failures=") &
                ~F.col("error_message").contains("decimal_failures=0"),
                "CONVERSÃO DECIMAL"
            )
            .when(
                F.col("error_message").contains("date_failures=") &
                ~F.col("error_message").contains("date_failures=0"),
                "CONVERSÃO DATE"
            )
            .when(
                F.col("error_message").contains("invalid_code_count=") &
                ~F.col("error_message").contains("invalid_code_count=0"),
                "CÓDIGO INVÁLIDO"
            )
            .otherwise("OUTRO")
        )
        .orderBy(
            "competencia_mov",
            "metric_group",
            "column_name",
            "metric_name"
        )
)

In [ ]:
# VALIDAÇÃO FINAL

if spark.catalog.tableExists(SILVER_IDENTIFIER):
    df_final = spark.table(SILVER_IDENTIFIER)
    print("\n========== SILVER ==========")
    print(f"Linhas Silver: {df_final.count():,}")
    df_final.printSchema()
    display(df_final.limit(10))

print("\n========== CONTROLE DA EXECUÇÃO ==========")
display(
    spark.table(CONTROL_IDENTIFIER)
         .filter(F.col("execution_id") == execution_id)
         .orderBy("competencia_mov", "metric_group", "column_name", "metric_name")
)